# Deliverable 2 — CalendarBot

Extracción estructurada de eventos de calendario desde lenguaje natural.

- **Modelo:** Qwen/Qwen3-4B-Instruct-2507
- **Fecha de referencia:** 2026-09-01 (martes)
- **Benchmark:** 45 frases (Benchmark2)
- **Intervención:** Prompt mejorado + post-procesador determinista (fechas relativas, duración, recordatorios)

In [ ]:
import json
import re
import os
import csv
from datetime import datetime, timedelta
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

FECHA_REFERENCIA = "2026-09-01"
DIA_REFERENCIA = "martes"
FECHA_REF_DT = datetime.strptime(FECHA_REFERENCIA, "%Y-%m-%d")
SEMILLA = 42

print(f"Fecha de referencia: {DIA_REFERENCIA} {FECHA_REFERENCIA}")
print(f"Semilla: {SEMILLA}")

Fecha de referencia: martes 2026-09-01
Semilla: 42


In [ ]:
# ============================================================
# CARGA DEL BENCHMARK
# ============================================================
BENCHMARK_FILE = "frases.json"

with open(BENCHMARK_FILE, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

frases_benchmark = benchmark.get("frases", [])
total_declarado = benchmark.get("metadata", {}).get("total_frases")
assert total_declarado == len(frases_benchmark), "El benchmark no coincide con su total declarado."

print(f"Benchmark: {BENCHMARK_FILE} | {len(frases_benchmark)} frases")
print(f"Categorías: {list(benchmark['metadata']['categorias'].keys())}")

Benchmark: frases.json | 45 frases
Categorías: ['simple', 'con_participantes', 'ambigua', 'compleja', 'edge_case', 'chat']


In [ ]:
# ============================================================
# CARGA DEL MODELO
# ============================================================
torch.manual_seed(SEMILLA)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEMILLA)

MODELO_ID = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODELO_ID, torch_dtype=torch.float16, device_map="auto"
)
print(f"Modelo: {MODELO_ID}")
print(f"Dispositivo: {model.device}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Modelo: Qwen/Qwen3-4B-Instruct-2507
Dispositivo: cuda:0


---
## 1. System Prompt mejorado
Reglas + few-shot examples (incluye ejemplos chat).

In [ ]:
SYSTEM_PROMPT = """Eres un extractor de datos de calendario.

REGLAS:
1. HOY ES {dia} {fecha}. Calcula fechas relativas desde esta fecha.
2. Si un campo NO se menciona, usa null (o [] para participantes). NUNCA inventes.
3. titulo: extrae SOLO la palabra clave de la actividad. Ej: "agéndame una junta" → "junta", "quedamos" → "Quedada", "schedule a call" → "call". Ignora muletillas (oye, porfa, q onda, acuerdate, avísame, puedes, agenda, agéndame).
4. fecha: YYYY-MM-DD. Para fechas RELATIVAS (mañana, el lunes, la próxima semana, etc.) usa null; el sistema las calcula automáticamente. Solo devuelve fecha absoluta si aparece un día y mes exactos ("el 15 de septiembre", "10/09"). hora_inicio: HH:MM 24h.
5. duracion_minutos: entero (1 hora=60). Si no hay = null.
6. participantes: lista de nombres reales. Si no hay = [].
7. ubicacion: donde se realiza. Si no se menciona = null.
8. recordatorio_minutos: entero. Si no se menciona = null. Si dice "avísame X min antes" o "recordar X min", usa X.
9. zona_horaria: "America/Santiago" por defecto; Madrid = "Europe/Madrid" y Argentina = "America/Argentina/Buenos_Aires".
10. Si se menciona virtual, videollamada o una plataforma, extraelo como ubicacion.
11. Si una fecha u hora es invalida, usa null en ese campo; no la corrijas inventando datos.

EJEMPLOS:

Frase: "Reunión con Ana el viernes a las 10am"
{{"titulo": "Reunión", "fecha": null, "hora_inicio": "10:00", "duracion_minutos": null, "participantes": ["Ana"], "ubicacion": null, "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}}

Frase: "Meeting on Friday at 10am"
{{"titulo": "Meeting", "fecha": null, "hora_inicio": "10:00", "duracion_minutos": null, "participantes": [], "ubicacion": null, "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}}

Frase: "Oye, agéndame una junta con la Dani pasado mañana a las 4 en la sala B-11"
{{"titulo": "junta", "fecha": null, "hora_inicio": "16:00", "duracion_minutos": null, "participantes": ["Dani"], "ubicacion": "sala B-11", "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}}

Frase: "agenda reunion mañana 10am porfa"
{{"titulo": "reunión", "fecha": null, "hora_inicio": "10:00", "duracion_minutos": null, "participantes": [], "ubicacion": null, "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}}

Frase: "q onda, quedamos el viernes a las 7 en lo de siempre?"
{{"titulo": "Quedada", "fecha": null, "hora_inicio": "19:00", "duracion_minutos": null, "participantes": [], "ubicacion": "lo de siempre", "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}}

Frase: "hey schedule a call with the team on Friday 3pm"
{{"titulo": "call", "fecha": null, "hora_inicio": "15:00", "duracion_minutos": null, "participantes": ["team"], "ubicacion": null, "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}}

Frase: "acuerdate que el lunes tenemos la presentación a las 9"
{{"titulo": "presentación", "fecha": null, "hora_inicio": "09:00", "duracion_minutos": null, "participantes": [], "ubicacion": null, "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}}

Frase: "puedes agendar la reunión de sincronización el 10 a las 14? avísame 15 min antes"
{{"titulo": "reunión de sincronización", "fecha": null, "hora_inicio": "14:00", "duracion_minutos": null, "participantes": [], "ubicacion": null, "recordatorio_minutos": 15, "zona_horaria": "America/Santiago"}}

Frase: "junta virtual mañana a las 5 con el profe"
{{"titulo": "junta", "fecha": null, "hora_inicio": "17:00", "duracion_minutos": null, "participantes": ["el profe"], "ubicacion": "virtual", "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}}

Frase: "reunión con Dani el jueves a las 11, 1 hora"
{{"titulo": "reunión", "fecha": null, "hora_inicio": "11:00", "duracion_minutos": 60, "participantes": ["Dani"], "ubicacion": null, "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}}

Frase: "Reunión"
{{"titulo": "Reunión", "fecha": null, "hora_inicio": null, "duracion_minutos": null, "participantes": [], "ubicacion": null, "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}}"""

def construir_prompt(frase_texto, usar_system=True):
    if usar_system:
        system = SYSTEM_PROMPT.format(dia=DIA_REFERENCIA, fecha=FECHA_REFERENCIA)
        user_msg = 'Frase: "' + frase_texto + '"\n\nRespuesta (SOLO JSON):'
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user_msg}
        ]
    else:
        user_msg = 'Frase: "' + frase_texto + '"\n\nRespuesta (SOLO JSON):'
        messages = [
            {"role": "user", "content": user_msg}
        ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

print("Prompt configurado")

Prompt configurado


---
## 2. Post-procesador determinista
Incluye: resolución de fechas relativas, normalización de horas, extracción de duración y recordatorios desde el texto.

In [ ]:
CAMPOS = ["titulo", "fecha", "hora_inicio", "duracion_minutos",
          "participantes", "ubicacion", "recordatorio_minutos",
          "zona_horaria"]

DIAS_SEMANA = {
    "lunes": 0, "martes": 1, "miércoles": 2, "miercoles": 2,
    "jueves": 3, "viernes": 4, "sábado": 5, "sabado": 5, "domingo": 6,
    "monday": 0, "tuesday": 1, "wednesday": 2, "thursday": 3,
    "friday": 4, "saturday": 5, "sunday": 6
}

MESES = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4,
    "mayo": 5, "junio": 6, "julio": 7, "agosto": 8,
    "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12
}


def extraer_json(respuesta):
    if not isinstance(respuesta, str):
        return {"error": "No JSON detected"}
    decoder = json.JSONDecoder()
    for match in re.finditer(r"\{", respuesta):
        try:
            objeto, _ = decoder.raw_decode(respuesta[match.start():])
            if isinstance(objeto, dict):
                return objeto
        except json.JSONDecodeError:
            continue
    return {"error": "No JSON detected", "raw": respuesta[:200]}


def resolver_fecha_relativa(texto, fecha_ref):
    texto_l = texto.lower()
    semana_offset = 0
    if "próxima semana" in texto_l or "proxima semana" in texto_l or "de la próxima semana" in texto_l:
        semana_offset = 7
    elif "en dos semanas" in texto_l:
        semana_offset = 14
    if "pasado mañana" in texto_l or "pasado manana" in texto_l:
        return (fecha_ref + timedelta(days=2)).strftime("%Y-%m-%d")
    if "mañana" in texto_l or "manana" in texto_l:
        return (fecha_ref + timedelta(days=1)).strftime("%Y-%m-%d")
    if semana_offset == 14:
        return (fecha_ref + timedelta(days=14)).strftime("%Y-%m-%d")
    patron_dias = "|".join(DIAS_SEMANA.keys())
    m = re.search(rf"(?:el\s+(?:próximo\s+|proximo\s+|que\s+viene\s+)?|on\s+)({patron_dias})", texto_l)
    if m:
        objetivo = DIAS_SEMANA[m.group(1)]
        actual = fecha_ref.weekday()
        diff = (objetivo - actual) % 7
        if diff == 0:
            diff = 7
        return (fecha_ref + timedelta(days=diff + semana_offset)).strftime("%Y-%m-%d")
    m = re.search(r"el\s+(\d{1,2})\s+(?:a\s+las|de|para)", texto_l)
    if m:
        dia = int(m.group(1))
        if 1 <= dia <= 31:
            return f"{fecha_ref.year}-{fecha_ref.month:02d}-{dia:02d}"
    m = re.search(r"(?:el\s+)?(\d{1,2})\s+de\s+(" + "|".join(MESES) + r")", texto_l)
    if m:
        return f"{fecha_ref.year}-{MESES[m.group(2)]:02d}-{int(m.group(1)):02d}"
    m = re.search(r"(\d{1,2})\s*/\s*(\d{1,2})", texto_l)
    if m:
        return f"{fecha_ref.year}-{int(m.group(2)):02d}-{int(m.group(1)):02d}"
    return None


def normalizar_fecha(valor):
    if isinstance(valor, str) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", valor):
        try:
            datetime.strptime(valor, "%Y-%m-%d")
            return valor
        except ValueError:
            pass
    return None


def normalizar_hora(valor, frase_texto):
    if not isinstance(valor, str):
        return None
    valor = valor.strip()
    m = re.fullmatch(r"(\d{1,2}):(\d{2})", valor)
    if m:
        h, mi = int(m.group(1)), int(m.group(2))
        if h <= 23 and mi <= 59:
            return f"{h:02d}:{mi:02d}"
        return None
    m = re.fullmatch(r"(\d{1,2})(?::(\d{2}))?\s*(am|pm)?", valor, re.I)
    if not m:
        return None
    hora = int(m.group(1))
    minuto = int(m.group(2) or 0)
    sufijo = (m.group(3) or "").lower()
    if sufijo == "pm" and hora < 12:
        hora += 12
    elif sufijo == "am" and hora == 12:
        hora = 0
    elif not sufijo:
        texto_l = frase_texto.lower()
        if "de la mañana" in texto_l or "de la manana" in texto_l:
            pass
        elif 1 <= hora <= 7:
            hora += 12
    if hora > 23 or minuto > 59:
        return None
    return f"{hora:02d}:{minuto:02d}"


def extraer_recordatorio(texto):
    texto_l = texto.lower()
    m = re.search(r"(?:avísame|avisame|recordar|recuérdame|recuerdame)\s+(\d+)\s*(?:min|minutos?)\s*(?:antes|before)?", texto_l)
    if m:
        return int(m.group(1))
    m = re.search(r"(\d+)\s*(?:min|minutos?)\s*antes", texto_l)
    if m:
        return int(m.group(1))
    return None


def extraer_duracion(texto):
    texto_l = texto.lower()
    tiene_recordatorio = bool(re.search(r"(?:avísame|avisame|recordar|recuérdame|recuerdame|antes)", texto_l))
    m = re.search(r"(\d+)\s*horas?\s*y\s*(?:una\s+)?media", texto_l)
    if m:
        return int(m.group(1)) * 60 + 30
    m = re.search(r"(\d+)\s*horas?\s*y\s*(\d+)\s*minutos?", texto_l)
    if m:
        return int(m.group(1)) * 60 + int(m.group(2))
    m = re.search(r"(\d+)\s*horas?", texto_l)
    if m:
        return int(m.group(1)) * 60
    if not tiene_recordatorio:
        m = re.search(r"(\d+)\s*(?:min|minutos?)", texto_l)
        if m:
            return int(m.group(1))
    return None


def post_procesar(json_salida, frase_texto):
    if not isinstance(json_salida, dict) or "error" in json_salida:
        return json_salida

    r = {c: json_salida.get(c, [] if c == "participantes" else None) for c in CAMPOS}

    r["fecha"] = normalizar_fecha(r.get("fecha"))
    if r["fecha"] is None:
        r["fecha"] = resolver_fecha_relativa(frase_texto, FECHA_REF_DT)

    r["hora_inicio"] = normalizar_hora(r.get("hora_inicio"), frase_texto)

    participantes = r.get("participantes")
    r["participantes"] = (
        [x.strip() for x in participantes
         if isinstance(x, str) and x.strip()
         and not x.startswith("<") and not x.strip().isdigit()]
        if isinstance(participantes, list) else []
    )

    dur_json = r.get("duracion_minutos")
    if isinstance(dur_json, (int, float)):
        r["duracion_minutos"] = int(dur_json)
    else:
        r["duracion_minutos"] = extraer_duracion(frase_texto)

    rec_json = r.get("recordatorio_minutos")
    if isinstance(rec_json, (int, float)):
        r["recordatorio_minutos"] = int(rec_json)
    else:
        r["recordatorio_minutos"] = extraer_recordatorio(frase_texto)

    texto = frase_texto.lower()
    if "madrid" in texto:
        r["zona_horaria"] = "Europe/Madrid"
    elif "argentina" in texto:
        r["zona_horaria"] = "America/Argentina/Buenos_Aires"
    elif not isinstance(r.get("zona_horaria"), str) or not r["zona_horaria"].strip():
        r["zona_horaria"] = "America/Santiago"

    if not r.get("ubicacion"):
        for etiqueta in ["virtual", "videollamada", "Google Meet", "Zoom", "Teams"]:
            if etiqueta.lower() in texto:
                r["ubicacion"] = etiqueta
                break

    return r


print("Post-procesador listo")

Post-procesador listo


---
## 3. Ejecución del pipeline
Corre el modelo sobre el benchmark completo.

In [ ]:
def ejecutar_pipeline(benchmark_frases, usar_system_prompt=True):
    resultados = []
    for frase in benchmark_frases:
        prompt = construir_prompt(frase["texto"], usar_system=usar_system_prompt)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=256, temperature=0.01,
                do_sample=True, top_p=0.95, pad_token_id=tokenizer.eos_token_id
            )

        respuesta = tokenizer.decode(
            outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True
        ).strip()
        json_salida = extraer_json(respuesta)
        json_salida = post_procesar(json_salida, frase["texto"])
        resultados.append({
            "id": frase["id"],
            "texto": frase["texto"],
            "resultado_modelo": json_salida
        })
    return resultados

In [ ]:
# ============================================================
# BASELINE: Prompting directo (sin system prompt ni few-shot)
# ============================================================
print("Ejecutando BASELINE (prompting directo)...")
resultados_baseline = ejecutar_pipeline(frases_benchmark, usar_system_prompt=False)
print(f"Baseline completado: {len(resultados_baseline)} frases")

Ejecutando BASELINE (prompting directo)...
Baseline completado: 45 frases


In [ ]:
import gc
import torch

# ============================================================
# SOLUCIÓN: Prompt mejorado + post-procesador
# ============================================================
print("Liberando memoria de la GPU...")
gc.collect()
torch.cuda.empty_cache()

print("Ejecutando SOLUCIÓN (prompt mejorado + post-procesador)...")
resultados_solucion = ejecutar_pipeline(frases_benchmark, usar_system_prompt=True)
print(f"Solución completada: {len(resultados_solucion)} frases")

Liberando memoria de la GPU...
Ejecutando SOLUCIÓN (prompt mejorado + post-procesador)...
Solución completada: 45 frases


---
## 4. Evaluación vs Ground Truth

In [ ]:
def evaluar(resultados, benchmark_frases):
    correctos = 0
    errores_campo = {c: 0 for c in CAMPOS}
    errores_tipo = {"inventado": 0, "omision": 0, "valor_incorrecto": 0}
    correctos_categoria = {}
    detalles = []

    for r in resultados:
        gt_frase = next((f for f in benchmark_frases if f["id"] == r["id"]), None)
        if gt_frase is None:
            continue
        gt = gt_frase["ground_truth"]
        pred = r["resultado_modelo"]
        cat = gt_frase.get("categoria", "unknown")
        correctos_categoria.setdefault(cat, {"total": 0, "correctos": 0})["total"] += 1

        if "error" in pred:
            for c in CAMPOS:
                errores_campo[c] += 1
                errores_tipo["valor_incorrecto"] += 1
            detalles.append({"id": r["id"], "texto": r["texto"], "exacto": False})
            continue

        ok_general = True
        for c in CAMPOS:
            esp = gt.get(c)
            opt = pred.get(c)
            field_ok = True
            if esp is None and opt is None:
                pass
            elif esp is None and opt is not None:
                errores_campo[c] += 1
                errores_tipo["inventado"] += 1
                ok_general = False
                field_ok = False
            elif esp is not None and opt is None:
                errores_campo[c] += 1
                errores_tipo["omision"] += 1
                ok_general = False
                field_ok = False
            elif isinstance(esp, list):
                pred_list = [str(x).lower() for x in (opt or [])]
                gt_list = [str(x).lower() for x in esp]
                if set(gt_list) != set(pred_list):
                    errores_campo[c] += 1
                    errores_tipo["valor_incorrecto"] += 1
                    ok_general = False
                    field_ok = False
            else:
                if str(esp).lower() != str(opt).lower():
                    errores_campo[c] += 1
                    errores_tipo["valor_incorrecto"] += 1
                    ok_general = False
                    field_ok = False

        if ok_general:
            correctos += 1
            correctos_categoria[cat]["correctos"] += 1
        detalles.append({"id": r["id"], "texto": r["texto"], "exacto": ok_general})

    return correctos, errores_campo, errores_tipo, correctos_categoria, detalles

In [ ]:
total = len(frases_benchmark)

# Evaluar baseline
c_base, ec_base, et_base, cc_base, _ = evaluar(resultados_baseline, frases_benchmark)
print(f"{'='*55}")
print(f"  BASELINE (prompting directo)")
print(f"  Precisión: {c_base}/{total} ({c_base/total*100:.1f}%)")
print(f"{'='*55}")
for c in CAMPOS:
    ok = total - ec_base[c]
    print(f"  {c:25s}: {ok}/{total} ({ok/total*100:.1f}%)")
print("\n  Por categoría:")
for cat, v in cc_base.items():
    p = v["correctos"] / v["total"] * 100 if v["total"] else 0
    print(f"  {cat:20s}: {v['correctos']}/{v['total']} ({p:.1f}%)")
print(f"\n  Errores: {et_base}")

print()

# Evaluar solución
c_sol, ec_sol, et_sol, cc_sol, det_sol = evaluar(resultados_solucion, frases_benchmark)
print(f"{'='*55}")
print(f"  SOLUCIÓN (prompt mejorado + post-procesador)")
print(f"  Precisión: {c_sol}/{total} ({c_sol/total*100:.1f}%)")
print(f"{'='*55}")
for c in CAMPOS:
    ok = total - ec_sol[c]
    print(f"  {c:25s}: {ok}/{total} ({ok/total*100:.1f}%)")
print("\n  Por categoría:")
for cat, v in cc_sol.items():
    p = v["correctos"] / v["total"] * 100 if v["total"] else 0
    print(f"  {cat:20s}: {v['correctos']}/{v['total']} ({p:.1f}%)")
print(f"\n  Errores: {et_sol}")

print(f"\n{'='*55}")
print(f"  MEJORA: {c_base/total*100:.1f}% → {c_sol/total*100:.1f}% (+{(c_sol-c_base)/total*100:.1f}pp)")
print(f"{'='*55}")

  BASELINE (prompting directo)
  Precisión: 0/45 (0.0%)
  titulo                   : 0/45 (0.0%)
  fecha                    : 39/45 (86.7%)
  hora_inicio              : 4/45 (8.9%)
  duracion_minutos         : 44/45 (97.8%)
  participantes            : 30/45 (66.7%)
  ubicacion                : 38/45 (84.4%)
  recordatorio_minutos     : 45/45 (100.0%)
  zona_horaria             : 45/45 (100.0%)

  Por categoría:
  simple              : 0/8 (0.0%)
  con_participantes   : 0/7 (0.0%)
  ambigua             : 0/7 (0.0%)
  compleja            : 0/7 (0.0%)
  edge_case           : 0/8 (0.0%)
  chat                : 0/8 (0.0%)

  Errores: {'inventado': 0, 'omision': 95, 'valor_incorrecto': 20}

  SOLUCIÓN (prompt mejorado + post-procesador)
  Precisión: 38/45 (84.4%)
  titulo                   : 42/45 (93.3%)
  fecha                    : 45/45 (100.0%)
  hora_inicio              : 45/45 (100.0%)
  duracion_minutos         : 45/45 (100.0%)
  participantes            : 41/45 (91.1%)
  ubicacion  

---
## 5. Análisis de casos de fallo

In [ ]:
print("CASOS DONDE LA SOLUCIÓN AÚN FALLA:")
print("=" * 60)
fallos = [d for d in det_sol if not d["exacto"]]
for f in fallos:
    frase_info = next((fr for fr in frases_benchmark if fr["id"] == f["id"]), None)
    pred = next((r["resultado_modelo"] for r in resultados_solucion if r["id"] == f["id"]), {})
    gt = frase_info["ground_truth"] if frase_info else {}
    print(f"\n  ID {f['id']} [{frase_info.get('categoria', '?')}]\n    \"{f['texto']}\"")
    for c in CAMPOS:
        esp = gt.get(c)
        opt = pred.get(c)
        if str(esp).lower() != str(opt).lower():
            print(f"    {c}: esperado={esp}, obtenido={opt}")

In [ ]:
# Ejecutar análisis de fallos detallado para ver la diferencia de valores
print("CASOS DE FALLO EN DETALLE:")
print("=" * 60)
fallos = [d for d in det_sol if not d["exacto"]]
for f in fallos:
    frase_info = next((fr for fr in frases_benchmark if fr["id"] == f["id"]), None)
    pred = next((r["resultado_modelo"] for r in resultados_solucion if r["id"] == f["id"]), {})
    gt = frase_info["ground_truth"] if frase_info else {}
    print(f"\nID {f['id']} [{frase_info.get('categoria', '?')}]\n  Frase: \"{f['texto']}\"")
    for c in CAMPOS:
        esp = gt.get(c)
        opt = pred.get(c)
        if isinstance(esp, list) or isinstance(opt, list):
            set_esp = set([str(x).lower() for x in (esp or [])])
            set_opt = set([str(x).lower() for x in (opt or [])])
            if set_esp != set_opt:
                print(f"    {c:20s}: esperado={esp}, obtenido={opt}")
        elif str(esp).lower() != str(opt).lower():
            print(f"    {c:20s}: esperado={esp}, obtenido={opt}")

CASOS DE FALLO EN DETALLE:

ID 11 [con_participantes]
  Frase: "Llamada con el cliente el 4 de septiembre a las 11"
    participantes       : esperado=['cliente'], obtenido=['el cliente']

ID 15 [con_participantes]
  Frase: "Review con los desarrolladores el 10 de septiembre a las 14:00"
    participantes       : esperado=['los desarrolladores'], obtenido=['desarrolladores']

ID 24 [compleja]
  Frase: "Presentación de tesis el jueves 10 de septiembre a las 10:00, 2 horas, con el comité evaluador, en el auditorio, avísame 30 minutos antes"
    participantes       : esperado=['comité evaluador'], obtenido=['el comité evaluador']

ID 26 [compleja]
  Frase: "Almuerzo de trabajo con los clientes el viernes 4 de septiembre a las 13:30 en el restaurante La Tabla por 1 hora y media"
    titulo              : esperado=Almuerzo de trabajo, obtenido=Almuerzo

ID 32 [edge_case]
  Frase: "Junta de emergencia"
    titulo              : esperado=Junta de emergencia, obtenido=junta

ID 33 [edge_case]


---
## 6. Guardar resultados

In [ ]:
# Guardar resultados
with open("resultados_baseline.json", "w", encoding="utf-8") as f:
    json.dump(resultados_baseline, f, indent=2, ensure_ascii=False)

with open("resultados_solucion.json", "w", encoding="utf-8") as f:
    json.dump(resultados_solucion, f, indent=2, ensure_ascii=False)

# Guardar reporte CSV
with open("reporte_comparativo.csv", "w", newline="", encoding="utf-8") as f:
    headers = ["id", "texto", "categoria", "exacto_baseline", "exacto_solucion"]
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    for i, (rb, rs) in enumerate(zip(resultados_baseline, resultados_solucion)):
        cat = frases_benchmark[i].get("categoria", "")
        writer.writerow({
            "id": rb["id"],
            "texto": rb["texto"],
            "categoria": cat,
            "exacto_baseline": det_sol[i]["exacto"] if i < len(det_sol) else False,
            "exacto_solucion": det_sol[i]["exacto"] if i < len(det_sol) else False
        })

print("Archivos guardados:")
print("  - resultados_baseline.json")
print("  - resultados_solucion.json")
print("  - reporte_comparativo.csv")

---
## 7. Agendar en Google Calendar (API)
Conecta directamente con Google Calendar para crear los eventos extraídos.

**Requisitos previos:**
1. Ir a [Google Cloud Console](https://console.cloud.google.com/)
2. Crear proyecto → Habilitar "Google Calendar API"
3. Credentials → Create OAuth 2.0 Client ID → Desktop App
4. Descargar el JSON → Subirlo como `credentials.json` al Colab

In [ ]:
# Instalar dependencias (solo la primera vez)
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
import os
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

SCOPES = ["https://www.googleapis.com/auth/calendar"]

def autenticar_calendar():
    """Autentica con Google Calendar API via OAuth2."""
    creds = None

    # Buscar token guardado
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)

    # Si no hay credenciales válidas, hacer login
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            if not os.path.exists("credentials.json"):
                print("ERROR: Sube 'credentials.json' al Colab")
                print("Descárgalo de: https://console.cloud.google.com/apis/credentials")
                return None
            flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
            creds = flow.run_local_console()

        # Guardar token para futuras ejecuciones
        with open("token.json", "w") as f:
            f.write(creds.to_json())

    service = build("calendar", "v3", credentials=creds)
    print("Autenticado con Google Calendar")
    return service


def listar_calendarios(service):
    """Lista los calendarios disponibles."""
    result = service.calendarList().list().execute()
    calendarios = result.get("items", [])
    print("\nCalendarios disponibles:")
    for i, cal in enumerate(calendarios):
        print(f"  [{i}] {cal.get('summary', 'Sin nombre')} ({cal.get('id')})")
    return calendarios

In [ ]:
def crear_evento(service, calendar_id, ev):
    """Crea un evento en Google Calendar."""
    fecha = ev.get("fecha")
    hora = ev.get("hora_inicio")
    titulo = ev.get("titulo", "Evento")
    duracion = ev.get("duracion_minutos")
    ubicacion = ev.get("ubicacion") or ""
    participantes = ev.get("participantes", [])
    recordatorio = ev.get("recordatorio_minutos")
    zona = ev.get("zona_horaria", "America/Santiago")

    evento_body = {"summary": titulo}

    # Ubicación
    if ubicacion:
        evento_body["location"] = ubicacion

    # Fecha y hora
    if fecha and hora:
        evento_body["start"] = {
            "dateTime": f"{fecha}T{hora}:00",
            "timeZone": zona
        }
        if duracion:
            h, m = map(int, hora.split(":"))
            total_min = h * 60 + m + duracion
            eh, em = divmod(total_min, 60)
            evento_body["end"] = {
                "dateTime": f"{fecha}T{eh:02d}:{em:02d}:00",
                "timeZone": zona
            }
        else:
            h = int(hora.split(":")[0])
            evento_body["end"] = {
                "dateTime": f"{fecha}T{h+1:02d}{hora[2:]}",
                "timeZone": zona
            }
    elif fecha:
        evento_body["start"] = {"date": fecha}
        evento_body["end"] = {"date": fecha}
    else:
        return None

    # Participantes
    if participantes:
        evento_body["attendees"] = [
            {"email": f"{p.lower().replace(' ', '.')}@placeholder.com"}
            for p in participantes
        ]

    # Recordatorio
    if recordatorio:
        evento_body["reminders"] = {
            "useDefault": False,
            "overrides": [
                {"method": "popup", "minutes": recordatorio},
                {"method": "email", "minutes": recordatorio}
            ]
        }

    evento = service.events().insert(
        calendarId=calendar_id,
        body=evento_body,
        sendUpdates="none"
    ).execute()

    return evento.get("htmlLink")

In [ ]:
# ============================================================
# AGENDAR EVENTOS EN GOOGLE CALENDAR
# ============================================================

# 1. Autenticar
service = autenticar_calendar()

if service:
    # 2. Listar calendarios
    calendarios = listar_calendarios(service)

    # 3. Usar el calendario principal (primero de la lista)
    calendar_id = calendarios[0]["id"] if calendarios else "primary"
    print(f"\nUsando calendario: {calendar_id}")

    # 4. Crear eventos
    print("\nCreando eventos...")
    eventos_creados = 0
    for r in resultados_solucion:
            ev = r["resultado_modelo"]
            if "error" in ev or (ev.get("fecha") is None and ev.get("hora_inicio") is None):
                continue
            try:
                link = crear_evento(service, calendar_id, ev)
                eventos_creados += 1
                print(f"  OK: {ev.get('titulo')} ({ev.get('fecha')} {ev.get('hora_inicio', '')})")
            except HttpError as e:
                print(f"  ERROR: {ev.get('titulo')} - {e}")

    print(f"\n{'='*50}")
    print(f"Eventos creados: {eventos_creados}/{len(resultados_solucion)}")
    print(f"Ve tu calendario: https://calendar.google.com")
    print(f"{'='*50}")
else:
    print("\nNo se pudo autenticar. Usando generación .ics como alternativa.")
    num_eventos, archivo = generar_ics(resultados_solucion)
    print(f"Archivo generado: {archivo}")
    print(f"Eventos: {num_eventos}")

---
## 8. Alternativa: Generar .ics (sin API)
Si no tienes `credentials.json`, genera un archivo `.ics` que puedes importar manualmente.

In [ ]:
from uuid import uuid4

def generar_ics(resultados, nombre_archivo="calendarbot_eventos.ics"):
    """Genera un archivo .ics con todos los eventos extraídos."""
    lineas = [
        "BEGIN:VCALENDAR",
        "VERSION:2.0",
        "PRODID:-//CalendarBot//Deliverable 2//ES",
        "CALSCALE:GREGORIAN",
        "METHOD:PUBLISH",
        "X-WR-CALNAME:CalendarBot Eventos",
        "X-WR-TIMEZONE:America/Santiago",
    ]

    eventos_creados = 0
    for r in resultados:
        ev = r["resultado_modelo"]
        if "error" in ev:
            continue
        if ev.get("fecha") is None and ev.get("hora_inicio") is None:
            continue

        uid = str(uuid4())
        titulo = ev.get("titulo", "Evento")
        fecha = ev.get("fecha")
        hora = ev.get("hora_inicio")
        duracion = ev.get("duracion_minutos")
        ubicacion = ev.get("ubicacion") or ""
        participantes = ev.get("participantes", [])
        recordatorio = ev.get("recordatorio_minutos")
        zona = ev.get("zona_horaria", "America/Santiago")

        if fecha and hora:
            dtstart = f"{fecha.replace('-', '')}T{hora.replace(':', '')}00"
        elif fecha:
            dtstart = f"{fecha.replace('-', '')}"
        else:
            continue

        if fecha and hora and duracion:
            h, m = map(int, hora.split(":"))
            total_min = h * 60 + m + duracion
            eh, em = divmod(total_min, 60)
            dtend = f"{fecha.replace('-', '')}T{eh:02d}{em:02d}00"
        elif fecha and hora:
            h, m = map(int, hora.split(":"))
            dtend = f"{fecha.replace('-', '')}T{(h+1):02d}{m:02d}00"
        else:
            dtend = None

        lineas.append("BEGIN:VEVENT")
        lineas.append(f"UID:{uid}")
        lineas.append("DTSTAMP:20260901T120000Z")
        lineas.append(f"DTSTART;TZID={zona}:{dtstart}")
        if dtend:
            lineas.append(f"DTEND;TZID={zona}:{dtend}")
        lineas.append(f"SUMMARY:{titulo}")
        if ubicacion:
            lineas.append(f"LOCATION:{ubicacion}")
        if participantes:
            for p in participantes:
                lineas.append(f"ATTENDEE;CN={p}:mailto:{p.lower().replace(' ', '.')}@placeholder.com")
        if recordatorio:
            lineas.append("BEGIN:VALARM")
            lineas.append("ACTION:DISPLAY")
            lineas.append(f"DESCRIPTION:Recordatorio: {titulo}")
            lineas.append(f"TRIGGER:-PT{recordatorio}M")
            lineas.append("END:VALARM")
        lineas.append("END:VEVENT")
        eventos_creados += 1

    lineas.append("END:VCALENDAR")

    with open(nombre_archivo, "w", encoding="utf-8") as f:
        f.write("\r\n".join(lineas))

    return eventos_creados, nombre_archivo


num_eventos, archivo = generar_ics(resultados_solucion)
print(f"Archivo .ics generado: {archivo}")
print(f"Eventos: {num_eventos}")
print(f"\nImportar en Google Calendar → Ajustes → Importar")

In [ ]:
# Previsualizar los eventos generados
print("EVENTOS AGENDADOS:")
print("=" * 70)
for r in resultados_solucion:
    ev = r["resultado_modelo"]
    if "error" in ev or (ev.get("fecha") is None and ev.get("hora_inicio") is None):
        continue
    fecha = ev.get("fecha", "sin fecha")
    hora = ev.get("hora_inicio", "--:--")
    titulo = ev.get("titulo", "?")
    dur = ev.get("duracion_minutos")
    ubic = ev.get("ubicacion", "")
    partic = ", ".join(ev.get("participantes", []))
    rec = ev.get("recordatorio_minutos")

    dur_str = f" ({dur}min)" if dur else ""
    loc_str = f" @ {ubic}" if ubic else ""
    part_str = f" con [{partic}]" if partic else ""
    rec_str = f" | Recordatorio: {rec}min" if rec else ""

    print(f"  {fecha} {hora}{dur_str} | {titulo}{part_str}{loc_str}{rec_str}")

print("=" * 70)
print(f"Total: {num_eventos} eventos")